## CDC_PROCESS Python Variant

Hieronder vind je de Python variant van de het CDC proces.


### Imports

In [181]:
import snowflake.connector
from snowflake.connector import DictCursor
from typing import Any, Optional
import os

### Connectie met Snowflake

In [ ]:
conn = snowflake.connector.connect(
    user="williamvdm",
    password="",
    account="rgqgoxq-nwb12951",
    warehouse="COMPUTE_WH",
    database="CDC_PYTHON_DB",
    role="SYSADMIN"
)



### Initaliseren
- Config ophalen
- Business kolommen ophalen

In [183]:


def initialise(config_id: int, v_runid: Any):
    with conn.cursor(DictCursor) as cur:
        # Config van de entiteit ophalen
        cur.execute(
            """
            SELECT ENTITY_NAME, PRIMARY_KEY_COLUMN, SOURCE_TABLE, TARGET_TABLE, DELETE_STRATEGY, ERROR_STRATEGY, UPDATE_STRATEGY
            FROM CDC.CDC_CONFIG
            WHERE CONFIG_ID = %s
              AND IS_ACTIVE = TRUE
            """,
            (config_id,),
        )
        cfg = cur.fetchone()

        v_entity = cfg["ENTITY_NAME"] if cfg else None
        if v_entity is None:
            cur.execute(
                """
                UPDATE LOGGING.RUN_LOG
                SET END_TS = CURRENT_TIMESTAMP(), STATUS = 'FAILED',
                    ERROR_CODE = 'CONFIG_NOT_FOUND', ERROR_MESSAGE = %s
                WHERE RUN_ID = %s
                """,
                (v_runid,),
            )
            conn.commit()
            return f"Error: config met id {config_id} niet gevonden of niet actief."

        v_pk = cfg["PRIMARY_KEY_COLUMN"]
        v_source = cfg["SOURCE_TABLE"]
        v_target = cfg["TARGET_TABLE"]
        v_delete_strategy = cfg["DELETE_STRATEGY"]
        v_error_strategy = cfg["ERROR_STRATEGY"]
        v_update_strategy = cfg["UPDATE_STRATEGY"]

        # Business kolommen van de entiteit ophalen (alle kolommen behalve kolommen die we gebruiken voor CDC logica: ROW_HASH, START_TS, END_TS, IS_ACTIVE, CDC_OPERATION en de primary key).
        cur.execute(
            """
            SELECT LISTAGG('"' || COLUMN_NAME || '"', ', ') AS BUSINESS_COLUMNS
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = 'STAGING'
              AND TABLE_NAME = UPPER(REGEXP_SUBSTR(%s, '[^.]+$'))
              AND COLUMN_NAME NOT IN ('ROW_HASH', 'START_TS', 'END_TS', 'IS_ACTIVE', 'CDC_OPERATION')
              AND COLUMN_NAME != %s
            """,
            (v_source, v_pk),
        )
        row = cur.fetchone()
        v_business_columns = row["BUSINESS_COLUMNS"] if row else None

        # ROW_HASH kolom toevoegen en vullen indien nodig
        cur.execute(f"ALTER TABLE {v_source} ADD COLUMN IF NOT EXISTS ROW_HASH STRING")
        cur.execute(
            f"""
            UPDATE {v_source}
            SET ROW_HASH = SHA2(TO_VARCHAR(OBJECT_CONSTRUCT(*)), 256)
            WHERE ROW_HASH IS NULL
            """
        )
        conn.commit()

    return {
        "v_entity": v_entity,
        "v_pk": v_pk,
        "v_source": v_source,
        "v_target": v_target,
        "v_delete_strategy": v_delete_strategy,
        "v_error_strategy": v_error_strategy,
        "v_update_strategy": v_update_strategy,
        "v_business_columns": v_business_columns,
    }

### Errors detecteren & loggen

In [184]:
def detect_errors(v_runid, init_values):
    v_entity = init_values["v_entity"]
    v_pk = init_values["v_pk"]
    v_source = init_values["v_source"]

    with conn.cursor() as cur:
        # Duplicate inserts: zelfde PK + zelfde hash
        v_sql = f"""
        INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
        SELECT
            {v_runid},
            '{v_entity}',
            'DUPLICATE_INSERT',
            OBJECT_CONSTRUCT(*)
        FROM (
            SELECT
                s.*,
                COUNT(*) OVER (PARTITION BY s.{v_pk}, s.ROW_HASH) AS CNT
            FROM {v_source} s
            WHERE s.{v_pk} IS NOT NULL
              AND s.{v_pk} != ''
              AND s.ROW_HASH IS NOT NULL
        ) t
        WHERE t.CNT > 1
        """
        cur.execute(v_sql)
        v_duplicate_inserts = cur.rowcount

        # Duplicate updates: zelfde PK + verschillende hash
        v_sql = f"""
        INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
        SELECT
            {v_runid},
            '{v_entity}',
            'DUPLICATE_UPDATE',
            OBJECT_CONSTRUCT(*)
        FROM (
            SELECT s.*
            FROM {v_source} s
            WHERE s.{v_pk} IS NOT NULL
              AND s.{v_pk} != ''
              AND s.ROW_HASH IS NOT NULL
              AND EXISTS (
                  SELECT 1
                  FROM {v_source} s2
                  WHERE s2.{v_pk} = s.{v_pk}
                    AND s2.ROW_HASH IS NOT NULL
                    AND s2.ROW_HASH <> s.ROW_HASH
              )
        ) t
        """
        cur.execute(v_sql)
        v_duplicate_updates = cur.rowcount

        # Key errors: NULL of lege PK
        v_sql = f"""
        INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
        SELECT
            {v_runid},
            '{v_entity}',
            'PRIMARY_KEY_ERROR',
            OBJECT_CONSTRUCT(*)
        FROM (
            SELECT s.*
            FROM {v_source} s
            WHERE s.{v_pk} IS NULL
               OR s.{v_pk} = ''
        ) t
        """
        cur.execute(v_sql)
        v_key_errors = cur.rowcount

    conn.commit()

    return {
        "v_duplicate_inserts": v_duplicate_inserts,
        "v_duplicate_updates": v_duplicate_updates,
        "v_key_errors": v_key_errors,
    }

### Inserts uitvoeren

In [185]:
def execute_inserts(v_runid: Any, init_values: dict):
    v_target = init_values["v_target"]
    v_pk = init_values["v_pk"]
    v_source = init_values["v_source"]
    v_business_columns = init_values["v_business_columns"] or ""

    business_cols = [c.strip() for c in v_business_columns.split(",") if c.strip()]
    business_cols_select = ", ".join([f"s.{c}" for c in business_cols])

    insert_cols_part = f"ROW_HASH, START_TS, IS_ACTIVE, CDC_OPERATION, {v_pk}" + (f", {v_business_columns}" if v_business_columns else "")
    select_cols_part = "s.ROW_HASH, CURRENT_TIMESTAMP(), TRUE, 'I', s." + v_pk + (f", {business_cols_select}" if business_cols_select else "")

    v_sql = f"""
    INSERT INTO {v_target} ({insert_cols_part})
    SELECT {select_cols_part}
    FROM {v_source} s
    LEFT JOIN {v_target} t
      ON t.{v_pk} = s.{v_pk} AND t.IS_ACTIVE = TRUE
    WHERE t.{v_pk} IS NULL
      AND s.{v_pk} IS NOT NULL
      AND s.{v_pk} != ''
      AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
    """

    with conn.cursor() as cur:
        cur.execute(v_sql)
        v_inserts = cur.rowcount
    conn.commit()
    return v_inserts

### Updates uitvoeren

In [186]:
def execute_updates(v_runid: Any, init_values: dict):
    v_target = init_values["v_target"]
    v_pk = init_values["v_pk"]
    v_source = init_values["v_source"]
    v_business_columns = init_values["v_business_columns"] or ""
    v_update_strategy = (init_values["v_update_strategy"] or "").upper()

    business_cols = [c.strip() for c in v_business_columns.split(",") if c.strip()]
    business_cols_select = ", ".join([f"s.{c}" for c in business_cols])
    business_cols_insert = f", {v_business_columns}" if v_business_columns else ""
    business_cols_select_insert = f", {business_cols_select}" if business_cols_select else ""
    business_cols_set = ", ".join([f"t.{c} = s.{c}" for c in business_cols])
    business_cols_set_part = f", {business_cols_set}" if business_cols_set else ""

    with conn.cursor() as cur:
        if v_update_strategy == "HISTORY":
            # Oude actieve versie sluiten
            v_sql = f"""
            UPDATE {v_target} t
            SET IS_ACTIVE = FALSE, END_TS = CURRENT_TIMESTAMP()
            WHERE t.IS_ACTIVE = TRUE
              AND EXISTS (
                SELECT 1
                FROM {v_source} s
                WHERE s.{v_pk} = t.{v_pk}
                  AND s.{v_pk} IS NOT NULL
                  AND s.{v_pk} != ''
                  AND s.ROW_HASH <> t.ROW_HASH
                  AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
              )
            """
            cur.execute(v_sql)

            # Nieuwe versie toevoegen als update-record
            v_sql = f"""
            INSERT INTO {v_target} (ROW_HASH, START_TS, IS_ACTIVE, CDC_OPERATION, {v_pk}{business_cols_insert})
            SELECT s.ROW_HASH, CURRENT_TIMESTAMP(), TRUE, 'U', s.{v_pk}{business_cols_select_insert}
            FROM {v_source} s
            WHERE s.{v_pk} IS NOT NULL
              AND s.{v_pk} != ''
              AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
              AND NOT EXISTS (
                SELECT 1
                FROM {v_target} t
                WHERE t.{v_pk} = s.{v_pk}
                  AND t.IS_ACTIVE = TRUE
                  AND t.ROW_HASH = s.ROW_HASH
              )
            """
        else:
            # OVERWRITE: actieve rij direct overschrijven
            v_sql = f"""
            UPDATE {v_target} t
            SET t.ROW_HASH = s.ROW_HASH,
                t.START_TS = CURRENT_TIMESTAMP(),
                t.IS_ACTIVE = TRUE,
                t.CDC_OPERATION = 'U',
                t.{v_pk} = s.{v_pk}
                {business_cols_set_part}
            FROM {v_source} s
            WHERE t.{v_pk} = s.{v_pk}
              AND s.{v_pk} IS NOT NULL
              AND s.{v_pk} != ''
              AND t.IS_ACTIVE = TRUE
              AND t.ROW_HASH <> s.ROW_HASH
              AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
            """

        cur.execute(v_sql)
        v_updates = cur.rowcount

    conn.commit()
    return v_updates

### Deletes uitvoeren

In [187]:
def execute_deletes(v_runid: Any, init_values: dict):
    v_target = init_values["v_target"]
    v_source = init_values["v_source"]
    v_pk = init_values["v_pk"]
    v_delete_strategy = (init_values["v_delete_strategy"] or "").upper()

    if v_delete_strategy == "SOFT":
        v_sql = f"""
        UPDATE {v_target} t
        SET IS_ACTIVE = FALSE,
            END_TS = CURRENT_TIMESTAMP(),
            CDC_OPERATION = 'D'
        WHERE t.IS_ACTIVE = TRUE
          AND NOT EXISTS (
            SELECT 1
            FROM {v_source} s
            WHERE s.{v_pk} = t.{v_pk}
          )
        """
    else:
        v_sql = f"""
        DELETE FROM {v_target} t
        WHERE t.IS_ACTIVE = TRUE
          AND NOT EXISTS (
            SELECT 1
            FROM {v_source} s
            WHERE s.{v_pk} = t.{v_pk}
          )
        """

    with conn.cursor() as cur:
        cur.execute(v_sql)
        v_deletes = cur.rowcount

    conn.commit()
    return v_deletes

### Logging

In [188]:

# with conn.cursor() as cur:
#     cur.execute(
#         """
#         INSERT INTO LOGGING.RUN_ENTITY_LOG (
#             RUN_ID,
#             START_TS,
#             END_TS,
#             ENTITY_NAME,
#             INSERT_COUNT,
#             UPDATE_COUNT,
#             DELETE_COUNT,
#             UNCHANGED_COUNT,
#             DUPLICATE_INSERT_COUNT,
#             DUPLICATE_UPDATE_COUNT,
#             KEY_ERROR_COUNT
#         )
#         VALUES (%s, %s, CURRENT_TIMESTAMP(), %s, %s, %s, %s, %s, %s, %s, %s)
#         """,
#         (
#             run_id,
#             v_start_ts,
#             v_entity,
#             v_inserts,
#             v_updates,
#             v_deletes,
#             v_unchanged,
#             v_duplicate_inserts,
#             v_duplicate_updates,
#             v_key_errors,
#         ),
#     )
# conn.commit()

### Uitvoeren van het proces

In [189]:
# Orchestratie
with conn.cursor() as cur:
    cur.execute("SELECT COALESCE(MAX(RUN_ID), 0) + 1 AS NEXT_RUN_ID FROM LOGGING.RUN_LOG")
    run_id = cur.fetchone()[0]

with conn.cursor() as cur:
    cur.execute(
        """
        INSERT INTO LOGGING.RUN_LOG (RUN_ID, START_TS, STATUS)
        VALUES (%s, CURRENT_TIMESTAMP(), 'RUNNING')
        """,
        (run_id,),
    )
conn.commit()

fatal_error = False

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT CONFIG_ID
        FROM CDC.CDC_CONFIG
        WHERE IS_ACTIVE = TRUE
        ORDER BY CONFIG_ID
        """
    )
    config_ids = [row[0] for row in cur.fetchall()]

for config_id in config_ids:
    try:
        v_start_ts = None
        with conn.cursor() as cur:
            cur.execute("SELECT CURRENT_TIMESTAMP()")
            v_start_ts = cur.fetchone()[0]

        init_values = initialise(config_id, run_id)
        if not isinstance(init_values, dict):
            # initialise gaf al foutafhandeling terug
            continue

        v_entity = init_values["v_entity"]
        v_source = init_values["v_source"]
        v_target = init_values["v_target"]
        v_pk = init_values["v_pk"]
        v_error_strategy = (init_values["v_error_strategy"] or "CONTINUE").upper()

        err_values = detect_errors(run_id, init_values)
        v_duplicate_inserts = err_values["v_duplicate_inserts"]
        v_duplicate_updates = err_values["v_duplicate_updates"]
        v_key_errors = err_values["v_key_errors"]

        v_inserts = execute_inserts(run_id, init_values)
        v_updates = execute_updates(run_id, init_values)
        v_deletes = execute_deletes(run_id, init_values)

        # Ongewijzigde rijen: actieve target rijen met dezelfde hash als source
        with conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT COUNT(*)
                FROM {v_target} t
                JOIN {v_source} s
                  ON s.{v_pk} = t.{v_pk}
                WHERE t.IS_ACTIVE = TRUE
                  AND s.{v_pk} IS NOT NULL
                  AND s.{v_pk} != ''
                  AND s.ROW_HASH = t.ROW_HASH
                  AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
                """
            )
            v_unchanged = cur.fetchone()[0]

        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO LOGGING.RUN_ENTITY_LOG (
                    RUN_ID,
                    START_TS,
                    END_TS,
                    ENTITY_NAME,
                    ROWS_INSERTED,
                    ROWS_UPDATED,
                    ROWS_DELETED,
                    ROWS_UNCHANGED,
                    DUPLICATE_INSERTS,
                    DUPLICATE_UPDATES,
                    KEY_ERRORS
                )
                VALUES (%s, %s, CURRENT_TIMESTAMP(), %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    run_id,
                    v_start_ts,
                    v_entity,
                    v_inserts,
                    v_updates,
                    v_deletes,
                    v_unchanged,
                    v_duplicate_inserts,
                    v_duplicate_updates,
                    v_key_errors,
                ),
            )
        conn.commit()

    except Exception as e:
        conn.rollback()
        fatal_error = True

        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
                VALUES (%s, %s, 'PROCESSING_ERROR', OBJECT_CONSTRUCT('message', %s, 'config_id', %s))
                """,
                (run_id, init_values["v_entity"] if "init_values" in locals() and isinstance(init_values, dict) else None, str(e), config_id),
            )
        conn.commit()

        if "init_values" in locals() and isinstance(init_values, dict):
            if (init_values.get("v_error_strategy") or "CONTINUE").upper() == "STOP":
                break

with conn.cursor() as cur:
    cur.execute(
        """
        UPDATE LOGGING.RUN_LOG
        SET END_TS = CURRENT_TIMESTAMP(),
            STATUS = %s
        WHERE RUN_ID = %s
        """,
        ("FAILED" if fatal_error else "SUCCESS", run_id),
    )
conn.commit()

print(f"CDC run afgerond. RUN_ID={run_id}, STATUS={'FAILED' if fatal_error else 'SUCCESS'}")

CDC run afgerond. RUN_ID=30, STATUS=SUCCESS
